# Chiffrement des donnees

Cellule utilitaire pour exporter une sauvegarde complete de la base `cours_securite` avec l'utilisateur `philippe`.

Si tu veux sauvegarder uniquement un schema, renseigne `schema_name` dans la cellule de code.

In [1]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import shutil
import subprocess

db_name = "cours_securite"
db_user = "philippe"
schema_name = None  # Exemple : "cours_securite" pour ne sauvegarder qu'un schema

backup_dir = Path("backups")
backup_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_label = schema_name if schema_name else db_name
backup_suffix = "schema" if schema_name else "database"
backup_file = backup_dir / f"{backup_label}_{backup_suffix}_{timestamp}.dump"

pg_dump_path = shutil.which("pg_dump")
if not pg_dump_path:
    raise FileNotFoundError(
        "pg_dump est introuvable. Verifie que PostgreSQL est installe et que pg_dump est dans le PATH."
    )

command = [
    pg_dump_path,
    "-U", db_user,
    "-d", db_name,
    "-F", "c",
    "-f", str(backup_file),
]

if schema_name:
    command[5:5] = ["-n", schema_name]

env = os.environ.copy()

result = subprocess.run(command, capture_output=True, text=True, env=env)

if result.returncode != 0 and "password" in (result.stderr or "").lower() and "PGPASSWORD" not in env:
    env["PGPASSWORD"] = getpass("Mot de passe PostgreSQL pour philippe : ")
    result = subprocess.run(command, capture_output=True, text=True, env=env)

if result.returncode != 0:
    print("Commande executee :", " ".join(command))
    print(result.stderr)
    raise RuntimeError(
        "La sauvegarde a echoue. Le message d'erreur pg_dump est affiche juste au-dessus."
    )

print(f"Sauvegarde terminee : {backup_file.resolve()}")
print("Restauration possible avec :")
print(f"pg_restore -U {db_user} -d {db_name} --clean '{backup_file.resolve()}'")


Sauvegarde terminee : /Users/philippe/Library/Mobile Documents/com~apple~CloudDocs/IA SCHOOL/04_DEV/03_Python/Projets/cours_securite/backups/cours_securite_database_20260419_090517.dump
Restauration possible avec :
pg_restore -U philippe -d cours_securite --clean '/Users/philippe/Library/Mobile Documents/com~apple~CloudDocs/IA SCHOOL/04_DEV/03_Python/Projets/cours_securite/backups/cours_securite_database_20260419_090517.dump'


In [2]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import shutil
import subprocess

db_name = "cours_securite"
db_user = "philippe"
csv_schemas = None  # None = detection automatique, ou liste comme ["production", "research"]

export_root = Path("csv_exports")
export_root.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_dir = export_root / f"{db_name}_tables_{timestamp}"
export_dir.mkdir(exist_ok=True)

psql_path = shutil.which("psql")
if not psql_path:
    raise FileNotFoundError(
        "psql est introuvable. Verifie que PostgreSQL est installe et que psql est dans le PATH."
    )

env = os.environ.copy()

def run_psql(sql: str):
    command = [
        psql_path,
        "-U", db_user,
        "-d", db_name,
        "-v", "ON_ERROR_STOP=1",
        "-At",
        "-c", sql,
    ]
    result = subprocess.run(command, capture_output=True, text=True, env=env)
    if result.returncode != 0 and "password" in (result.stderr or "").lower() and "PGPASSWORD" not in env:
        env["PGPASSWORD"] = getpass(f"Mot de passe PostgreSQL pour {db_user} : ")
        result = subprocess.run(command, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print("Commande executee :", " ".join(command))
        print(result.stderr)
        raise RuntimeError("Execution psql en echec. Le message d'erreur est affiche juste au-dessus.")
    return result.stdout

def quote_ident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

if csv_schemas is None:
    schemas_sql = (
        "SELECT schemaname "
        "FROM pg_catalog.pg_tables "
        "WHERE schemaname NOT IN ('pg_catalog', 'information_schema') "
        "GROUP BY schemaname "
        "ORDER BY schemaname;"
    )
    schemas = [line.strip() for line in run_psql(schemas_sql).splitlines() if line.strip()]
elif isinstance(csv_schemas, str):
    schemas = [csv_schemas]
else:
    schemas = list(csv_schemas)

if not schemas:
    raise RuntimeError("Aucun schema utilisateur contenant des tables n'a ete trouve.")

exported_count = 0

for schema_name in schemas:
    escaped_schema = schema_name.replace("'", "''")
    tables_sql = (
        "SELECT tablename "
        "FROM pg_catalog.pg_tables "
        f"WHERE schemaname = '{escaped_schema}' "
        "ORDER BY tablename;"
    )
    tables = [line.strip() for line in run_psql(tables_sql).splitlines() if line.strip()]
    if not tables:
        print(f"Aucune table a exporter dans le schema {schema_name!r}.")
        continue

    schema_dir = export_dir / schema_name
    schema_dir.mkdir(exist_ok=True)

    for table_name in tables:
        csv_file = schema_dir / f"{table_name}.csv"
        copy_sql = (
            f"COPY {quote_ident(schema_name)}.{quote_ident(table_name)} "
            "TO STDOUT WITH CSV HEADER"
        )
        csv_content = run_psql(copy_sql)
        csv_file.write_text(csv_content, encoding="utf-8")
        exported_count += 1
        print(f"Exporte : {csv_file.resolve()}")

if exported_count == 0:
    raise RuntimeError("Aucune table n'a pu etre exportee en CSV.")

print(f"{exported_count} table(s) exportee(s).")
print(f"Export CSV termine dans : {export_dir.resolve()}")


Exporte : /Users/philippe/Library/Mobile Documents/com~apple~CloudDocs/IA SCHOOL/04_DEV/03_Python/Projets/cours_securite/csv_exports/cours_securite_tables_20260419_090524/production/clients.csv
Exporte : /Users/philippe/Library/Mobile Documents/com~apple~CloudDocs/IA SCHOOL/04_DEV/03_Python/Projets/cours_securite/csv_exports/cours_securite_tables_20260419_090524/production/transactions.csv
Exporte : /Users/philippe/Library/Mobile Documents/com~apple~CloudDocs/IA SCHOOL/04_DEV/03_Python/Projets/cours_securite/csv_exports/cours_securite_tables_20260419_090524/research/algos_trading.csv
3 table(s) exportee(s).
Export CSV termine dans : /Users/philippe/Library/Mobile Documents/com~apple~CloudDocs/IA SCHOOL/04_DEV/03_Python/Projets/cours_securite/csv_exports/cours_securite_tables_20260419_090524
